# NMDAR / GluN2B inhibitor activity prediction

This notebook presents the reproducible machine-learning workflow for the NMDAR (GluN2B) inhibitor project. It loads the prepared descriptor and fingerprint datasets, checks the train/validation/test partitions, trains representative classifiers, evaluates predictions on the held-out test set, and provides a reusable prediction helper.

**Scope:** molecular feature generation is represented by the prepared project datasets in this repository.

The binary target is `Activity`, where the project data encodes active and inactive compounds as numeric classes.

## Project files and data split

The notebook expects these files to be in the same directory as the notebook:

- `final_ml_dataset_train.csv` model fitting data
- `final_ml_dataset_validation.csv` validation data
- `final_ml_dataset_test.csv`  final held-out evaluation data
- `Model_Performance_Comparison.csv` consolidated project benchmark results

The prepared feature table contains 249 scaled descriptor columns and RDKit, Morgan, and MACCS fingerprint columns.

In [ ]:
from pathlib import Path

data_path = Path(
    r"C:\Users\omen\OneDrive - Shri Vile Parle Kelavani Mandal\Documents\NMDAR_ML\data_raw.csv"
)

df = pd.read_csv(data_path, low_memory=False)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

PROJECT_DIR = Path.cwd()
TARGET = "Activity"

def require_file(filename: str) -> Path:
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Required project file was not found: {path}. "
            "Run this notebook from the repository directory."
        )
    return path

train = pd.read_csv(require_file("final_ml_dataset_train.csv"))
validation = pd.read_csv(require_file("final_ml_dataset_validation.csv"))
test = pd.read_csv(require_file("final_ml_dataset_test.csv"))

for name, frame in (("train", train), ("validation", validation), ("test", test)):
    if TARGET not in frame.columns:
        raise ValueError(f"{TARGET!r} is missing from the {name} dataset")

X_train, y_train = train.drop(columns=TARGET), train[TARGET].astype(int)
X_validation, y_validation = validation.drop(columns=TARGET), validation[TARGET].astype(int)
X_test, y_test = test.drop(columns=TARGET), test[TARGET].astype(int)
if list(X_train.columns) != list(X_validation.columns) or list(X_train.columns) != list(X_test.columns):
    raise ValueError("Train, validation, and test feature columns must match in the same order")

print(f"Train: {train.shape}; validation: {validation.shape}; test: {test.shape}")
print(f"Feature count: {X_train.shape[1]}")
display(pd.DataFrame({"split": ["train", "validation", "test"], "active": [int((y_train == 1).sum()), int((y_validation == 1).sum()), int((y_test == 1).sum())], "inactive": [int((y_train == 0).sum()), int((y_validation == 0).sum()), int((y_test == 0).sum())]}))

In [ ]:
feature_groups = {
    "scaled_descriptors": [c for c in X_train.columns if c.startswith("Scaled_")],
    "rdkit_fingerprints": [c for c in X_train.columns if c.startswith("RDKit_")],
    "morgan_fingerprints": [c for c in X_train.columns if c.startswith("Morgan_")],
    "maccs_fingerprints": [c for c in X_train.columns if c.startswith("MACCS_")],
}
display(pd.DataFrame({"feature_group": list(feature_groups), "features": [len(v) for v in feature_groups.values()]}))

In [ ]:
# Step 1: Remove exact duplicate rows
df_cleaned = df.drop_duplicates().copy()
after_duplicates = len(df_cleaned)
print("After duplicate removal:", after_duplicates)

# Step 2: Remove mixtures based on ligand name
mixture_pattern = r"mixture|racemic"

mask_mixture = (
    df_cleaned["BindingDB Ligand Name"]
    .astype("string")
    .str.contains(mixture_pattern, case=False, na=False, regex=True)
)

df_cleaned = df_cleaned.loc[~mask_mixture].copy()
after_mixture_removal = len(df_cleaned)

print("After mixture removal:", after_mixture_removal)

In [ ]:
# Replace the summary Line with the following to include the count of removed mixtures
print("After duplicate removal:", len(df_cleaned) +
      int(mask_mixture.sum()))
# with
print("After duplicate removal:", after_duplicates)
#You can also add the requested audit table after creating
audit_df = pd.DataFrame([
    {
        "Step": "Initial dataset",
        "Rows": len(df),
        "Active": 0,
        "Inactive": 0,
    },
    {
        "Step": "After exact duplicate removal",
        "Rows": after_duplicates,
        "Active": 0,
        "Inactive": 0,
    },
    {
        "Step": "After mixture removal",
        "Rows": after_mixture_removal,
        "Active": 0,
        "Inactive": 0,
    },
    {
        "Step": "Rows with reported IC50",
        "Rows": len(df_ic50),
        "Active": 0,
        "Inactive": 0,
    },
    {
        "Step": "Rows with numeric IC50",
        "Rows": int(df_ic50["IC50_numeric"].notna().sum()),
        "Active": 0,
        "Inactive": 0,
    },
    {
        "Step": "Final labelled dataset",
        "Rows": len(df_filtered),
        "Active": active,
        "Inactive": inactive,
    },
])

print("\nFiltering audit:")
print(audit_df.to_string(index=False))

audit_df.to_csv("dataset_filtering_audit.csv", index=False)

## Representative models

The descriptor model uses logistic regression because the descriptor columns are already scaled. The fingerprint model uses a random forest, which is suitable for sparse binary fingerprints and produces class probabilities for downstream ranking. Validation data is used for a sanity check; the test set remains the final held-out evaluation.

In [ ]:
def evaluate(model, X, y, label):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else None
    result = {
        "model": label,
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities) if probabilities is not None else np.nan,
    }
    print(label)
    print(classification_report(y, predictions, target_names=["Inactive", "Active"], zero_division=0))
    display(pd.DataFrame([result]))
    display(pd.DataFrame(confusion_matrix(y, predictions), index=["Actual inactive", "Actual active"], columns=["Predicted inactive", "Predicted active"]))
    return result

descriptor_columns = feature_groups["scaled_descriptors"]
morgan_columns = feature_groups["morgan_fingerprints"]

# Descriptor model
sc = StandardScaler()
X_mordred = X_train[descriptor_columns]
X_mordred_normalized = sc.fit_transform(X_mordred)
model_mordred = RandomForestClassifier(
    bootstrap=True, random_state=42, max_depth=10,
    min_samples_split=4, n_estimators=500, n_jobs=-1,
)
model_mordred.fit(X_mordred_normalized, y_train)
evaluate(model_mordred, sc.transform(X_validation[descriptor_columns]), y_validation, "Random forest / Mordred descriptors")

# KRFP is not present in the prepared project table. Train this block only when KRFP columns are supplied.
krfp_columns = [c for c in X_train.columns if c.startswith("KRFP_")]
model_krfp = None
if krfp_columns:
    X_krfp = X_train[krfp_columns]
    model_krfp = SVC(C=10, gamma=0.01, kernel="rbf", probability=True)
    model_krfp.fit(X_krfp, y_train)
    evaluate(model_krfp, X_validation[krfp_columns], y_validation, "SVC / KRFP fingerprints")
else:
    print("KRFP model skipped: no KRFP_ columns are present in the prepared dataset.")

# MACCS fingerprint model
maccs_columns = feature_groups["maccs_fingerprints"]
X_maccs = X_train[maccs_columns]
model_maccs = RandomForestClassifier(
    bootstrap=True, random_state=42, max_depth=15,
    min_samples_split=5, n_estimators=500, n_jobs=-1,
)
model_maccs.fit(X_maccs, y_train)
evaluate(model_maccs, X_validation[maccs_columns], y_validation, "Random forest / MACCS fingerprints")

# PubChem fingerprint model
pubchem_columns = [c for c in X_train.columns if c.startswith("PubChem_")]
model_pubchem = None
if pubchem_columns:
    X_pubchem = X_train[pubchem_columns]
    model_pubchem = RandomForestClassifier(
        bootstrap=True, random_state=42, max_depth=25,
        min_samples_split=5, n_estimators=800, n_jobs=-1,
    )
    model_pubchem.fit(X_pubchem, y_train)
    evaluate(model_pubchem, X_validation[pubchem_columns], y_validation, "Random forest / PubChem fingerprints")
else:
    print("PubChem model skipped: no PubChem_ columns are present in the prepared dataset.")

# Keep the existing prediction helper compatible with the Mordred and Morgan models.
descriptor_model = model_mordred
morgan_model = RandomForestClassifier(
    bootstrap=True, random_state=42, max_depth=25,
    min_samples_split=5, n_estimators=500, class_weight="balanced", n_jobs=-1,
)
morgan_model.fit(X_train[morgan_columns], y_train)
evaluate(morgan_model, X_validation[morgan_columns], y_validation, "Random forest / Morgan fingerprints")

## Final held-out test evaluation

These metrics are calculated only after model fitting and validation checks. They should be reported together with the project benchmark table rather than treated as a replacement for independent experimental validation.

In [ ]:
test_results = pd.DataFrame([
    evaluate(descriptor_model, sc.transform(X_test[descriptor_columns]), y_test, "Test: random forest / Mordred descriptors"),
    evaluate(morgan_model, X_test[morgan_columns], y_test, "Test: random forest / Morgan fingerprints"),
])
test_results

## Existing project benchmark

The repository also contains the consolidated benchmark generated during model comparison. Loading it here preserves the projectÃƒÂ¢Ã¢â€šÂ¬Ã¢â€žÂ¢s documented results without rerunning every hyperparameter search.

In [ ]:
benchmark = pd.read_csv(require_file("Model_Performance_Comparison.csv"))
benchmark.sort_values("ROC-AUC", ascending=False).head(10)

## Prediction helper

Pass one row containing the same prepared feature columns as the training data. The helper returns the predicted class and active-class probability from both representative models.

In [ ]:
def predict_activity(feature_row: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in X_train.columns if c not in feature_row.columns]
    if missing:
        raise ValueError(f"Prediction input is missing {len(missing)} prepared feature columns")
    row = feature_row.loc[:, X_train.columns]
    return pd.DataFrame({
        "descriptor_prediction": descriptor_model.predict(sc.transform(row[descriptor_columns])),
        "descriptor_active_probability": descriptor_model.predict_proba(sc.transform(row[descriptor_columns]))[:, 1],
        "morgan_prediction": morgan_model.predict(row[morgan_columns]),
        "morgan_active_probability": morgan_model.predict_proba(row[morgan_columns])[:, 1],
    })

predict_activity(X_test.iloc[[0]])

## Reproducibility notes

- Run the notebook from the  directory so the relative data paths resolve correctly.
- Keep the train/validation/test partitions fixed when comparing models.